# HS4002 Week 2

## Setup

Make sure this notebook and `gss2022_mini.csv` are in the same folder. Install packages below if needed.

In [ ]:
# !pip install pandas numpy plotnine

import pandas as pd
import numpy as np
import plotnine as p9
import warnings
warnings.filterwarnings('ignore')

# Python Fundamentals

Before working with real survey data, it's worth getting comfortable with the building blocks every Python program is made of: variables, strings, lists, dictionaries, conditionals, loops, and functions. Each of these will show up again, in a pandas-flavoured form, later in this notebook.

## Variables and Data Types

Create three variables: your name (a `str`), your age (an `int`), and your height in meters (a `float`). Print each one alongside its data type.

Hint: `type(some_variable)` tells you what data type something is.

In [ ]:
name = "Jamie"
age = 20
height = 1.68

print(name, type(name))

## Conditionals (if / elif / else)

Write a conditional that decides whether a respondent counts as a "minor", "young adult", or "adult" based on their age.

Hint: Python uses `if`, `elif`, `else` — and indentation (not curly braces) marks what belongs inside each branch.

In [ ]:
respondent_age = 17

if respondent_age <  :
    category = "minor"
elif respondent_age <  :
    category = "young adult"
else:
    category = "adult"

print(category)

## Functions

Wrap the minor / young adult / adult logic from earlier into a reusable **function**, then apply it to every age in `ages`.

Hint: `def function_name(parameter):` starts a function; `return` sends a value back to whoever called it.

In [ ]:
def categorize_age(age_input):
    if age_input < 18:
        return "minor"
    elif age_input  < 30:
        return "young adult"
    else:
        return "adult"

categorize_age()

## Objects, Attributes, and Methods

Everything in Python is an **object** — including the DataFrames you're about to work with. An object bundles together data (**attributes**, accessed with `object.attribute`, no parentheses) and behavior (**methods**, accessed with `object.method()`, always with parentheses — even if empty).

You'll see this pattern constantly for the rest of this notebook: `df.shape` and `df.columns` are attributes; `df.dropna()` and `df.sample()` are methods.

In [ ]:
example_list = [3, 1, 2]

print(len(example_list))     # a function, not a method
example_list.sort()          # a method -- changes the list in place
print(example_list)

example_str = "hello"
print(example_str.upper())   # another method, even on a plain string

## Reading the Data

Hint: use `pd.read_csv()`. The data object in pandas is called a **DataFrame**.

## How many rows does the file contain?

Hint: `len(df)` or `df.shape[0]`.

## Print the names of all columns

# Selecting and Sampling

Narrow the dataset down to five variables: survey ID (`id`), employment status (`wrkstat`), occupational prestige (`prestg10`), age (`age`), and gender (`sex`).

Hint: index the DataFrame with a list of column names.

In [ ]:
chosen_variables = []

df2 = df[chosen_variables].copy()

## Checking for Missing Data First

Before dropping any rows, it's worth seeing *how much* is actually missing, and in which columns — dropping blindly can silently throw away more data than you realize.

Hint: `.isna()` marks each cell `True`/`False` for missing; `.sum()` on top of that counts how many `True`s per column.

In [ ]:
print(df2.isna().sum())

## Dropping rows with missing values

Hint: use `.dropna()`.

In [ ]:
df2 = df2.dropna()

## Taking a sample of 500

Setting a random seed makes the sample reproducible.

Hint: use `.sample(n=..., random_state=...)`.

In [ ]:
df_mini = df2.sample(n=500, random_state=47)

## Sanity check — should be 500 rows and 5 columns

In [ ]:
print(df_mini.shape)

## Saving Your Cleaned Data

Write `df_mini` out to a new file, so you (or someone else) can pick up from here without re-running everything above.

Hint: `.to_csv(..., index=False)`. Name the output differently from the input, so you don't overwrite your source data.

In [ ]:
df_mini.to_csv("gss2022_mini_cleaned.csv", index=False)

# Making New Variables

Recode `wrkstat` into a working/not-working binary. Full-time (1), part-time (2), and with-a-job-but-not-at-work (3) are all counted as working.

Hint: `np.where(condition, if_true, if_false)` lets you recode based on a condition.

In [ ]:
df_mini = df_mini.assign(
    working  = np.where(df_mini['wrkstat'] <= 3, 'Working', 'Not Working'),
    sex_woman = np.where(df_mini['sex'] == 2, 'Woman', 'Not Woman')
)

## Sanity check — print the breakdown of each new variable

In [ ]:
print(df_mini['working'].value_counts())
print(df_mini['sex_woman'].value_counts())

## Casting a Column's Type

Sometimes you need to change a column's *type* without changing its values — different from recoding, which is what `sex_woman` did above.

Hint: `.astype(str)` casts a column to text.

In [ ]:
df_mini['sex_str'] = df_mini['sex'].astype(str)
print(df_mini['sex_str'].dtype)

# Producing a Table

We want the mean occupational prestige among working men and women.

## Filter for working individuals

Hint: use boolean indexing.

In [ ]:
df_wrk = df_mini.loc[df_mini['working'] == 'Working',:]

## Filtering on More Than One Condition

Narrow `df_mini` down to just the working women.

Hint: combining conditions in pandas uses `&` (and) / `|` (or) — not Python's plain `and`/`or` — and each condition needs its own parentheses.

In [ ]:
df_working_women = df_mini[
    (df_mini['working'] == 'Working') & (df_mini['sex_woman'] == 'Woman')
]
print(df_working_women.shape)

## Mean occupational prestige by gender

We'll do it the long way — split into two dataframes first (useful for learning), then retrieve the mean of each.

In [ ]:
df_men   = df_wrk[df_wrk['sex_woman'] == 'Not Woman']
df_women = df_wrk[df_wrk['sex_woman'] == 'Woman']

print(df_men['prestg10'].mean())
print(df_women['prestg10'].mean())

## Building a summary table

In [ ]:
first_table = pd.DataFrame({
    'gender':          ['Men', 'Women'],
    'mean_occ_prestg': [df_men['prestg10'].mean(), df_women['prestg10'].mean()]
})
first_table

## Exporting Your Table for Use in Word

pandas can save your table as a CSV file. A CSV stores table data in a format that Excel and Google Sheets can easily open.

After running the next cell:

1. Find the new CSV file in the same folder as this notebook.
2. Open it in Excel or import it into Google Sheets.
3. Select and copy the table.
4. Paste the table into your Word document.

Opening the CSV directly in Word is not recommended because Word may show it as plain, comma-separated text.

Hint: `.to_csv(..., index=False)` prevents pandas from adding an unnecessary column of row numbers.

In [ ]:
first_table.to_csv(
    "mean_occupational_prestige_by_gender.csv",
    index=False,
    float_format="%.1f"
)

In [ ]:
# Optional: export the table as LaTeX code for a LaTeX document
print(first_table.to_latex(
    index=False,
    float_format="%.1f",
    caption="Mean Occupational Prestige by Gender",
    header=["Gender", "Mean Occupational Prestige"]
))

# Making a Scatterplot

Plot the relationship between age and occupational prestige, coloured by gender.

In [ ]:
(
    p9.ggplot(df_mini) +
    p9.geom_point(p9.aes(x='age', y='prestg10', color='sex_woman'), alpha=0.5) +
    p9.theme_light() +
    p9.labs(x='Age', y='Occupational Prestige', color='Gender')
)